# Treino no Google Colab: detecção de toxicidade no chat de LoL

Roda no Colab as mesmas etapas de `src/train.py` que rodariam localmente:

- **Entrada:** a pasta `PLN/` no seu Google Drive (código, dados, cache dos folds e resultados já prontos).
- **Saída:** resultados e modelos salvos direto em `MyDrive/PLN/PLN/outputs/`.

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)*.

Se a sessão cair, rode tudo de novo: os folds já concluídos ficam salvos no Drive e são pulados.

## 1. GPU e Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

## 2. Copiar código e dados para o disco local (rápido); resultados continuam no Drive

In [ ]:
import os
DRIVE_BASE = '/content/drive/MyDrive/PLN'       # pasta PLN enviada para o Drive
LOCAL_BASE = '/content/PLN'

assert os.path.exists(f'{DRIVE_BASE}/PLN/src/train.py'), 'Envie a pasta PLN para a raiz do Meu Drive'
!mkdir -p {LOCAL_BASE}
!rsync -a --exclude outputs {DRIVE_BASE}/ {LOCAL_BASE}/

os.environ['PLN_OUTPUTS'] = f'{DRIVE_BASE}/PLN/outputs'   # resultados e modelos -> Drive
os.environ['PLN_CHECKPOINTS'] = '/content/checkpoints'     # checkpoints temporários -> disco da VM
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['PYTHONUNBUFFERED'] = '1'
os.makedirs(os.environ['PLN_OUTPUTS'], exist_ok=True)
%cd {LOCAL_BASE}/PLN/src

## 3. Dependências (o Colab já traz o PyTorch com CUDA)

In [ ]:
!pip -q install -r ../requirements.txt
import torch, transformers, sklearn
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| transformers', transformers.__version__, '| sklearn', sklearn.__version__)

## 4. Conferir dados e folds

Os folds vêm do cache (`cache/folds_*_k5.csv`), então são exatamente os mesmos do treino local.
Esta célula precisa terminar com `OK: nenhuma partida compartilhada...`.

In [ ]:
!python data.py

## 5. Validação cruzada agrupada por partida (folds prontos são pulados)

In [ ]:
!python train.py --model bert --folds 5 --protocol group

In [ ]:
!python train.py --model roberta --folds 5 --protocol group

## 6. Modelos finais (execução única no split oficial)

Salva o modelo completo em `outputs/<modelo>/best_model/`, junto com uma ficha (`README.md`) com as métricas.

In [ ]:
for m in ['distilbert', 'bert', 'roberta']:
    !python train.py --model {m}

## 7. Avaliação comparativa no teste oficial (métricas por subgrupo e predições)

In [ ]:
!python evaluate.py

## 8. Resumo

In [ ]:
import json, pathlib
out = pathlib.Path(os.environ['PLN_OUTPUTS'])
for m in ['distilbert', 'bert', 'roberta']:
    for p in ['cv_group', 'cv_message']:
        f = out / m / p / 'cv_summary.json'
        if f.exists():
            s = json.loads(f.read_text())
            print(f"{m:10s} {p:10s} F1 = {s['test_mean']['f1']:.4f} ± {s['test_std']['f1']:.4f}")
!du -sh {out}/*/best_model

## 9. Playground: testar mensagens nos três modelos

**Para só usar o playground, sem treinar de novo:** rode as células 1, 2 e 3 e depois pule direto para esta seção.
Ela carrega os modelos de `MyDrive/PLN/PLN/outputs/<modelo>/best_model`.

O Gradio gera um link público (`*.gradio.live`, válido por 72 h) que dá para mandar para os colegas.

In [ ]:
!pip -q install gradio
import sys
sys.path.insert(0, f'{LOCAL_BASE}/PLN/src')
import gradio as gr
import pandas as pd
from inference import load_models

MODELS = load_models(include_baseline=True)   # 3 modelos ajustados + toxic-bert sem fine-tuning

def avaliar(mensagem, limiar):
    texts = [t.strip() for t in mensagem.splitlines() if t.strip()]
    if not texts:
        return pd.DataFrame(), ''
    linhas, resumo = [], []
    for m in MODELS:
        t = m.default_threshold if m.baseline else limiar
        for texto, p in zip(texts, m.predict_proba(texts)):
            linhas.append({'mensagem': texto, 'modelo': m.name, 'prob. tóxica': round(p, 4),
                           'veredito': '⚠ tóxica' if p >= t else '✓ não tóxica'})
    df = pd.DataFrame(linhas)
    ajustados = df[~df['modelo'].str.contains('sem fine-tuning')]
    for texto, g in ajustados.groupby('mensagem', sort=False):
        n = (g['veredito'] == '⚠ tóxica').sum()
        resumo.append(f"- **{texto}**: {n} de {len(g)} modelos ajustados consideram tóxica")
    return df, chr(10).join(resumo)

demo = gr.Interface(
    fn=avaliar,
    inputs=[gr.Textbox(lines=4, label='Mensagem(ns) de chat em inglês (uma por linha)'),
            gr.Slider(0.05, 0.95, value=0.5, step=0.05, label='Limiar dos modelos ajustados')],
    outputs=[gr.Dataframe(label='Probabilidade por modelo'), gr.Markdown(label='Consenso')],
    examples=[['gg wp', 0.5], ['uninstall the game you trash noob', 0.5], ['STUPID FUCKING LAG', 0.5],
              ['report jungle he is feeding', 0.5], ['bg', 0.5]],
    title='Toxicidade no chat de LoL',
    description='DistilBERT, BERT e RoBERTa ajustados no L2DTnH, comparados ao toxic-bert sem fine-tuning (limiar 0,45).',
    flagging_mode='never',
)
demo.launch(share=True, debug=False)